In [ ]:
#"""ChromaDB Setup for RAG Chatbot (Module 1)"""

import os
import chromadb
from chromadb.utils import embedding_functions
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import google.generativeai as genai
import shutil

In [ ]:
# ============================================================================
# 1. SETUP
# ============================================================================

# Load environment variables
import dotenv
dotenv.load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini API configured")

In [ ]:
# ============================================================================
# 2. LOAD POLICY DOCUMENTS
# ============================================================================

POLICY_DOCS_PATH = '../data/raw/acko_policies/'

# Create directory if not exists
os.makedirs(POLICY_DOCS_PATH, exist_ok=True)

print("📚 Loading policy documents...")

# For demo, let's create sample policy content if PDFs don't exist
sample_policy_content = """
ACKO MOTOR INSURANCE POLICY - TERMS AND CONDITIONS

SECTION 1: COVERAGE
1.1 Comprehensive Coverage: 
The policy covers loss or damage to the insured vehicle due to:
- Accident
- Fire, lightning, explosion
- Theft, riot, strike
- Natural disasters (flood, earthquake, cyclone)

1.2 Third Party Liability:
The policy covers legal liability for third party death, injury, or property damage up to ₹10,00,000.

SECTION 2: ADD-ON COVERS
2.1 Zero Depreciation Cover:
Available at additional premium. No depreciation applied to parts replaced during claims.

2.2 Road Side Assistance:
24/7 roadside assistance including:
- Towing up to 50 km
- Flat tyre assistance
- Fuel delivery (up to 5 litres)

2.3 NCB Protection:
Protects your accumulated No Claim Bonus even after one claim.

SECTION 3: EXCLUSIONS
Loss or damage due to:
- Wear and tear
- Electrical/mechanical breakdown
- Driving under influence
- Unauthorized driving
- War or nuclear risks

SECTION 4: CLAIMS PROCESS
1. Report the incident immediately
2. File the claim online
3. Submit required documents
4. Get your vehicle assessed
5. Claim settlement within 7 days

ACKO HEALTH INSURANCE POLICY

SECTION 1: COVERAGE
1.1 Inpatient Hospitalization:
Covers expenses for hospitalization including:
- Room rent
- ICU charges
- Surgeon fees
- Medicines and consumables

1.2 Pre-existing Conditions:
Covered after 48 months of continuous coverage.

SECTION 2: WAITING PERIODS
2.1 Initial Waiting Period: 30 days from policy inception
2.2 Pre-existing Disease Waiting: 48 months
2.3 Specific Disease Waiting: 24 months (for certain conditions)

SECTION 3: ADD-ON COVERS
3.1 Maternity Cover:
Covers delivery and maternity expenses (9 months waiting)
3.2 Critical Illness Cover:
Lump sum payment for critical illness diagnosis

SECTION 4: CLAIMS PROCESS
1. Inform at the time of hospitalization
2. Submit pre-authorization request
3. Get claim processed within 3 days
4. Direct settlement with network hospitals

ACKO TRAVEL INSURANCE POLICY

SECTION 1: COVERAGE
1.1 Medical Expenses:
Covers emergency medical expenses abroad up to $50,000
1.2 Trip Cancellation:
Covers non-refundable expenses due to trip cancellation

SECTION 2: ADD-ON COVERS
2.1 Adventure Sports:
Covers injuries from adventure sports (additional premium)
2.2 Baggage Loss:
Covers loss of checked baggage up to ₹50,000

SECTION 3: EXCLUSIONS
- Pre-existing conditions
- Mental health conditions
- Self-inflicted injuries
- Alcohol/drug related incidents
"""

# Create sample PDF files if they don't exist
from fpdf import FPDF

def create_sample_pdf(content, filename):
    """Create a sample PDF with given content"""
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    
    # Split content into lines and add to PDF
    for line in content.split('\n'):
        if line.strip():
            pdf.multi_cell(0, 10, line)
    pdf.output(filename)

# Create sample PDFs
sample_files = [
    ('Acko_Motor_Policy.pdf', sample_policy_content.split('ACKO HEALTH INSURANCE POLICY')[0]),
    ('Acko_Health_Policy.pdf', sample_policy_content.split('ACKO TRAVEL INSURANCE POLICY')[0].split('ACKO HEALTH INSURANCE POLICY')[1]),
    ('Acko_Travel_Policy.pdf', sample_policy_content.split('ACKO TRAVEL INSURANCE POLICY')[1])
]

for filename, content in sample_files:
    filepath = os.path.join(POLICY_DOCS_PATH, filename)
    if not os.path.exists(filepath):
        create_sample_pdf(content, filepath)
        print(f"  Created: {filename}")

In [ ]:
# ============================================================================
# 3. LOAD AND CHUNK DOCUMENTS
# ============================================================================

def load_and_chunk_documents(pdf_paths):
    """Load PDFs and split into chunks"""
    
    all_documents = []
    
    for pdf_path in pdf_paths:
        print(f"📄 Loading: {pdf_path}")
        
        try:
            loader = PyPDFLoader(pdf_path)
            documents = loader.load()
            
            # Split into chunks
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=500,
                chunk_overlap=100,
                length_function=len,
                separators=["\n\n", "\n", ".", " ", ""]
            )
            
            chunks = text_splitter.split_documents(documents)
            
            # Add metadata
            for chunk in chunks:
                chunk.metadata['source'] = pdf_path
                
            all_documents.extend(chunks)
            print(f"  Created {len(chunks)} chunks")
            
        except Exception as e:
            print(f"  ❌ Error loading {pdf_path}: {e}")
    
    return all_documents

# Get all PDFs
pdf_files = [os.path.join(POLICY_DOCS_PATH, f) for f in os.listdir(POLICY_DOCS_PATH) if f.endswith('.pdf')]
chunks = load_and_chunk_documents(pdf_files)

print(f"\n✅ Total chunks created: {len(chunks)}")

In [ ]:
# ============================================================================
# 4. SETUP CHROMADB
# ============================================================================

# Delete existing ChromaDB if exists
CHROMA_PATH = '../data/chromadb'
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)
    print(f"🗑️ Removed existing ChromaDB at {CHROMA_PATH}")

# Initialize ChromaDB
client = chromadb.PersistentClient(path=CHROMA_PATH)

# Use free embedding function
embedding_function = embedding_functions.GoogleGenerativeAiEmbeddingFunction(
    api_key=GEMINI_API_KEY,
    model_name="models/embedding-001"
)

# Create or get collection
collection_name = "acko_policies"

# Delete if exists
try:
    client.delete_collection(collection_name)
except:
    pass

collection = client.create_collection(
    name=collection_name,
    embedding_function=embedding_function
)

print(f"✅ Created ChromaDB collection: {collection_name}")

In [ ]:
# ============================================================================
# 5. ADD DOCUMENTS TO CHROMADB
# ============================================================================

# Prepare documents for ChromaDB
documents = [chunk.page_content for chunk in chunks]
metadatas = [chunk.metadata for chunk in chunks]
ids = [f"chunk_{i}" for i in range(len(chunks))]

# Add to ChromaDB in batches
batch_size = 100
for i in range(0, len(documents), batch_size):
    end = min(i + batch_size, len(documents))
    
    collection.add(
        documents=documents[i:end],
        metadatas=metadatas[i:end],
        ids=ids[i:end]
    )
    
    print(f"  Added chunks {i+1} to {end}")

print(f"\n✅ Added {len(documents)} chunks to ChromaDB")

In [ ]:
# ============================================================================
# 6. TEST RETRIEVAL
# ============================================================================

def test_retrieval(query, n_results=3):
    """Test retrieval from ChromaDB"""
    
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    
    print(f"\nQuery: {query}")
    print("-" * 60)
    
    for i, (doc, metadata) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
        print(f"\nResult {i+1}:")
        print(f"Source: {metadata['source']}")
        print(f"Content: {doc[:200]}...")
    
    return results

# Test queries
test_queries = [
    "Does my insurance cover flood damage?",
    "What is zero depreciation cover?",
    "How do I file a claim?",
    "Is there a waiting period for health insurance?",
    "What does travel insurance cover?"
]

for query in test_queries:
    test_retrieval(query, n_results=2)

print("\n✅ ChromaDB setup complete!")
print(f"📊 Collection contains {collection.count()} chunks")

In [ ]:
# ============================================================================
# 7. SAVE COLLECTION INFO
# ============================================================================

collection_info = {
    'name': collection_name,
    'count': collection.count(),
    'embedding_function': 'GoogleGenerativeAiEmbeddingFunction'
}

import json
with open('../data/chromadb/collection_info.json', 'w') as f:
    json.dump(collection_info, f)

print("\n✅ ChromaDB setup complete and ready for use!")